# Phase 0: Finetuning the "Resume Expert" (LoRA)

Welcome to the training phase! In this notebook, we will teach a general-purpose LLM (Large Language Model) a new, specific skill: **Gap Analysis**.

### The Goal
We want our model to take two inputs:
1. A snippet from your resume.
2. A snippet from a job description.

And produce one output:
- A professional analysis of what skills are missing + new, targeted bullet points in LaTeX format.

### The Method: Parameter-Efficient Fine-Tuning (PEFT)
Training a 7-billion parameter model from scratch requires massive compute. Instead, we will use **LoRA (Low-Rank Adaptation)**. 

Think of LoRA like adding a small "post-it note" of new knowledge on top of a massive encyclopedia. We freeze the encyclopedia (the base model) and only train the post-it note (the adapter). This allows us to run training on a consumer GPU (like in Colab or a decent local machine).

### Step 1: Install Dependencies
We need a specific set of libraries to handle modern LLM training:
* `transformers`: The backbone library by Hugging Face to load models.
* `peft`: Library for LoRA and other efficiency techniques.
* `bitsandbytes`: Allows us to load the model in 4-bit precision (drastically reducing RAM usage).
* `trl`: Transformer Reinforcement Learning library, used here for Supervised Fine-Tuning (SFT).

In [ ]:
!pip install -q -U torch transformers peft datasets bitsandbytes trl

### Step 2: Imports & Configuration
Here we import the necessary modules. 

**Key Decisions:**
* **Base Model**: We are using `Mistral-7B-Instruct`. It's a very strong open-source model that follows instructions well.
* **Dataset**: We point to the `dataset.jsonl` file we generated earlier.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
    logging,
)

try:
    from transformers import BitsAndBytesConfig
    BNB_AVAILABLE = True
except ImportError:
    BitsAndBytesConfig = None
    BNB_AVAILABLE = False

from peft import LoraConfig, PeftModel
from trl import SFTTrainer

# Model from Hugging Face Hub
base_model_name = "01-ai/Yi-6B"

# New LoRA adapter name (this is what we will save)
new_model_name = "resume-expert-lora"

# Dataset path
dataset_path = "dataset.jsonl"

### Step 3: Load and Format the Data
LLMs don't natively understand JSON objects. They understand text. 

We need a **formatting function** that takes our structured JSON (Resume, JD, Analysis) and stitches it into a single prompt string. This format effectively tells the model: "When you see this Instruction and Context, you should generate this Output."

In [ ]:
dataset = load_dataset("json", data_files=dataset_path, split="train")

# formatting_func: Converts a data sample into a prompt string
def format_instruction(sample):
    return f"""### Instruction:
Analyze the resume gap based on the job description and provide specific bullet points in LaTeX format.

### Job Description:
{sample['job_description_context']}

### Resume Context:
{sample['resume_context']}

### Output Analysis:
{sample['output_analysis']}
"""

# Let's sanity check one example to see what the model actually sees
print(format_instruction(dataset[0]))

### Step 4: Quantization (Making it fit)
A standard 7B model requires ~14GB of VRAM (FP16). Training it requires even more. 

To make this accessible, we use **4-bit Quantization** (via `BitsAndBytesConfig`). This compresses the model weights significantly with minimal loss in performance. 
* `load_in_4bit=True`: Enables the compression.
* `bnb_4bit_quant_type="nf4"`: Normal Float 4 (a data type optimized for neural network weights).

In [ ]:
compute_dtype = getattr(torch, "float16")

if BNB_AVAILABLE and torch.cuda.is_available():
    print("Using 4-bit quantization via bitsandbytes (CUDA detected).")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=quant_config,
        device_map="auto"
    )
else:
    quant_config = None
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print("bitsandbytes not available or CUDA missing; loading model without 4-bit quantization.")
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=dtype,
        device_map="auto"
    )

model.config.use_cache = False # Silence warnings during training
model.config.pretraining_tp = 1

# Load the tokenizer (converts text to numbers)
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16 training issues

### Step 5: LoRA Configuration (The Adapter)
This is where we define our "post-it note" brain.

* `r=64`: The "rank" of the update matrices. Higher rank = more parameters to train = "smarter" adapter, but more compute. 64 is a healthy balance.
* `lora_alpha=16`: Scaling factor. Think of this as the "learning rate" multiplier for the adapter weights.
* `task_type="CAUSAL_LM"`: We are doing standard text generation.

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

### Step 6: Training Hyperparameters
These settings control how the model learns:
* `num_train_epochs=1`: How many times we show the entire dataset to the model. For finetuning, 1-3 epochs is usually enough. Overfitting is a risk.
* `learning_rate=2e-4`: How big of a step the optimizer takes. LoRA typically uses a higher LR than full fine-tuning.
* `per_device_train_batch_size=4`: How many examples to process at once. Lower this if you run out of memory.

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

### Step 7: Start Training
We pass everything to the `SFTTrainer` (Supervised Fine-Tuning Trainer) and hit go.
The trainer will:
1. Apply the LoRA config to the Base Model.
2. Tokenize the dataset using our `format_instruction`.
3. Run the training loop.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text", # This is dummy field for SFTTrainer, we rely on formatting_func
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
    formatting_func=format_instruction, 
)

trainer.train()

# Save the trained adapter
trainer.model.save_pretrained(new_model_name)

### Step 8: Inference (Did it work?)
Now that we have a trained adapter, let's test it.
We provide a prompt *without* the Output Analysis, and see if the model can generate it.

In [ ]:
prompt = """### Instruction:
Analyze the resume gap based on the job description and provide specific bullet points in LaTeX format.

### Job Description:
Looking for a Python dev with FastAPI experience.

### Resume Context:
Experienced in Python and Django.

### Output Analysis:
"""

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>{prompt}")
print(result[0]['generated_text'])